In [ ]:
import pandas as pd

# Load all three raw datasets — resumes needs utf-8 + a fix for double-encoded characters
resumes = pd.read_csv("../data/raw/resumes/resumes.csv", encoding="utf-8")
naukri = pd.read_csv("../data/raw/naukri/naukri.csv")
linkedin = pd.read_csv("../data/raw/linkedin/linkedin.csv")

# Fix the double-encoding corruption in resume text
def fix_encoding(text):
    try:
        return text.encode("latin-1", errors="ignore").decode("utf-8", errors="ignore")
    except:
        return text

resumes["Resume"] = resumes["Resume"].apply(fix_encoding)

print("Resumes shape:", resumes.shape)
print("Sample fixed text:")
print(resumes["Resume"].iloc[0][:200])

: 

In [ ]:
print("Number of unique categories:", resumes["Category"].nunique())
print("\nCategory counts:")
print(resumes["Category"].value_counts())

print("\nSample resume text:")
print(resumes["Resume"].iloc[0][:300])

In [ ]:
# Check resume text readability
print("Sample resume text:")
print(repr(resumes["Resume"].iloc[0][:400]))

# Check for missing values in each dataset
print("\nMissing values - Resumes:\n", resumes.isnull().sum())
print("\nMissing values - Naukri:\n", naukri.isnull().sum())
print("\nMissing values - LinkedIn:\n", linkedin.isnull().sum())

# Check for duplicate rows
print("\nDuplicate resumes:", resumes.duplicated().sum())
print("Duplicate naukri postings:", naukri.duplicated().sum())
print("Duplicate linkedin postings:", linkedin.duplicated().sum())

In [ ]:
# Drop duplicate resumes
print("Before dedup:", resumes.shape)
resumes = resumes.drop_duplicates().reset_index(drop=True)
print("After dedup:", resumes.shape)

# Fill missing skills in naukri with empty string (can't drop 45% of rows)
naukri["skills"] = naukri["skills"].fillna("")

# Confirm no more missing values
print("\nNaukri missing after fix:\n", naukri.isnull().sum())

In [ ]:
# Standardize naukri to common schema
naukri_clean = naukri.rename(columns={"job-description": "description"})
naukri_clean = naukri_clean[["title", "company", "location", "skills", "description", "experience"]]
naukri_clean["source"] = "naukri"

# Standardize linkedin to common schema (missing fields filled with empty string)
linkedin_clean = linkedin.copy()
linkedin_clean["company"] = ""
linkedin_clean["location"] = ""
linkedin_clean["experience"] = ""
linkedin_clean = linkedin_clean[["title", "company", "location", "skills", "description", "experience"]]
linkedin_clean["source"] = "linkedin"

# Combine into one job corpus
job_corpus = pd.concat([naukri_clean, linkedin_clean], ignore_index=True)

print("Job corpus shape:", job_corpus.shape)
print("\nSource breakdown:\n", job_corpus["source"].value_counts())
print("\nSample rows:")
job_corpus.head(3)

In [ ]:
naukri_clean = naukri.rename(columns={"job-description": "description"})
naukri_clean = naukri_clean[["title", "company", "location", "skills", "description", "experience"]]
naukri_clean["source"] = "naukri"

linkedin_clean = linkedin.copy()
linkedin_clean["company"] = ""
linkedin_clean["location"] = ""
linkedin_clean["experience"] = ""
linkedin_clean = linkedin_clean[["title", "company", "location", "skills", "description", "experience"]]
linkedin_clean["source"] = "linkedin"

job_corpus = pd.concat([naukri_clean, linkedin_clean], ignore_index=True)

print("Job corpus shape:", job_corpus.shape)
print("\nSource breakdown:\n", job_corpus["source"].value_counts())
print("\nSample rows:")
job_corpus.head(3)

In [ ]:
job_corpus.to_csv("../data/interim/job_corpus.csv", index=False)
print("Saved job corpus:", job_corpus.shape)

In [ ]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("Original resume snippet:", resumes["Resume"].iloc[0][:150])
print("\nCleaned:", clean_text(resumes["Resume"].iloc[0])[:150])

In [ ]:
resumes["clean_text"] = resumes["Resume"].apply(clean_text)
job_corpus["clean_text"] = (
    job_corpus["title"].fillna("") + " " +
    job_corpus["description"].fillna("") + " " +
    job_corpus["skills"].fillna("")
).apply(clean_text)

print("Resumes cleaned. Sample:")
print(resumes["clean_text"].iloc[0][:150])

print("\nJob corpus cleaned. Sample:")
print(job_corpus["clean_text"].iloc[0][:150])

# Save cleaned versions
resumes.to_csv("../data/processed/resumes_clean.csv", index=False)
job_corpus.to_csv("../data/processed/job_corpus_clean.csv", index=False)
print("\nSaved cleaned datasets to data/processed/")

In [ ]:
import sys
sys.path.append("../src")

from data.load_data import load_raw_datasets
from data.preprocess import clean_resumes, build_job_corpus, clean_text

resumes_raw, naukri_raw, linkedin_raw = load_raw_datasets("../data/raw")
resumes_test = clean_resumes(resumes_raw)
job_corpus_test = build_job_corpus(naukri_raw, linkedin_raw)

print("Sanity check - resumes:", resumes_test.shape)
print("Sanity check - job corpus:", job_corpus_test.shape)

In [ ]:
print("Resumes with missing/empty Resume text:", resumes_test["Resume"].isna().sum())
print("Resumes with missing Category:", resumes_test["Category"].isna().sum())